# Consumer Complaints Gold\n\nThis notebook mirrors `databricks/src/gold/consumer_complaints_gold.py` so the Gold warehouse build can be reviewed and run directly inside Databricks.

In [ ]:
"""Build the Gold Consumer Complaints dimensional model.\n\nThis job transforms the complaint-level Silver table into a small analytical\nwarehouse model suitable for dashboards, BI, and downstream feature engineering.\nThe implementation keeps one fact row per complaint and publishes supporting\ndimensions for date, product, company, and state.\n\nGold processing flow:\n1. Validate that the Silver source table exists and contains rows.\n2. Create the Gold schema if needed.\n3. Build reusable dimensions from the cleaned Silver table.\n4. Join dimensions back to Silver to create the complaint fact table.\n5. Overwrite all Gold tables idempotently.\n6. Validate row counts, uniqueness, and referential completeness.\n"""\n\nfrom __future__ import annotations\n\nimport logging\n\nfrom pyspark.sql import DataFrame, SparkSession\nfrom pyspark.sql import functions as F\nfrom pyspark.sql import types as T\nfrom pyspark.sql.window import Window\n\n\nCATALOG = "fintech_lakehouse_dev"\nSILVER_SCHEMA = "silver"\nGOLD_SCHEMA = "gold"\n\nSOURCE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_consumer_complaints"\n\nDIM_DATE_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_date"\nDIM_PRODUCT_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_product"\nDIM_COMPANY_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_company"\nDIM_STATE_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_state"\nFACT_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.fact_consumer_complaints"\n\n\nlogging.basicConfig(\n    level=logging.INFO,\n    format="%(asctime)s %(levelname)s %(message)s",\n)\n\nLOGGER = logging.getLogger(__name__)\n\nMISSING_MEMBER_LABEL = "NOT_PROVIDED"\n\n\n# ------------------------------------------------------------\n# Validate the cleaned Silver table before building any Gold\n# assets. Gold should only run when the complaint-level Silver\n# table exists and contains rows.\n# ------------------------------------------------------------\ndef validate_source_table(spark: SparkSession) -> tuple[DataFrame, int]:\n    """Validate that the Silver source table exists and contains records."""\n\n    if not spark.catalog.tableExists(SOURCE_TABLE):\n        raise RuntimeError(f"Silver source table does not exist: {SOURCE_TABLE}")\n\n    silver_dataframe = spark.table(SOURCE_TABLE)\n    silver_row_count = silver_dataframe.count()\n\n    if silver_row_count == 0:\n        raise RuntimeError(f"Silver source table is empty: {SOURCE_TABLE}")\n\n    LOGGER.info(\n        "Validated Silver source table %s with %s rows.",\n        SOURCE_TABLE,\n        f"{silver_row_count:,}",\n    )\n    return silver_dataframe, silver_row_count\n\n\n# ------------------------------------------------------------\n# Ensure the Gold schema exists so dimensions and facts can be\n# written as managed Unity Catalog Delta tables in a consistent\n# location.\n# ------------------------------------------------------------\ndef ensure_gold_schema_exists(spark: SparkSession) -> None:\n    """Create the Gold schema if it is missing."""\n\n    spark.sql(\n        f"""\n        CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}\n        COMMENT 'Business-ready dimensional warehouse for consumer complaints analytics.'\n        """\n    )\n\n\n# ------------------------------------------------------------\n# Build a conformed date dimension from all complaint received\n# and company-response dates. The dimension also includes an\n# explicit fallback row for unmatched or missing dates.\n# ------------------------------------------------------------\ndef build_dim_date(silver_dataframe: DataFrame) -> DataFrame:\n    """Build a conformed date dimension from received and sent complaint dates."""\n\n    all_dates = (\n        silver_dataframe.select(F.to_date("date_received").alias("full_date"))\n        .union(\n            silver_dataframe.select(F.to_date("date_sent_to_company").alias("full_date"))\n        )\n        .filter(F.col("full_date").isNotNull())\n        .distinct()\n    )\n\n    dim_date = (\n        all_dates\n        .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast(T.IntegerType()))\n        .withColumn("calendar_year", F.year("full_date"))\n        .withColumn("calendar_quarter", F.quarter("full_date"))\n        .withColumn("calendar_month", F.month("full_date"))\n        .withColumn("calendar_month_name", F.date_format("full_date", "MMMM"))\n        .withColumn("calendar_day", F.dayofmonth("full_date"))\n        .withColumn("day_of_week", F.dayofweek("full_date"))\n        .withColumn("day_name", F.date_format("full_date", "EEEE"))\n        .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))\n    )\n\n    fallback_row = spark_single_row(\n        silver_dataframe.sparkSession,\n        {\n            "date_key": 0,\n            "full_date": None,\n            "calendar_year": None,\n            "calendar_quarter": None,\n            "calendar_month": None,\n            "calendar_month_name": MISSING_MEMBER_LABEL,\n            "calendar_day": None,\n            "day_of_week": None,\n            "day_name": MISSING_MEMBER_LABEL,\n            "is_weekend": None,\n        },\n        schema=T.StructType(\n            [\n                T.StructField("date_key", T.IntegerType(), False),\n                T.StructField("full_date", T.DateType(), True),\n                T.StructField("calendar_year", T.IntegerType(), True),\n                T.StructField("calendar_quarter", T.IntegerType(), True),\n                T.StructField("calendar_month", T.IntegerType(), True),\n                T.StructField("calendar_month_name", T.StringType(), True),\n                T.StructField("calendar_day", T.IntegerType(), True),\n                T.StructField("day_of_week", T.IntegerType(), True),\n                T.StructField("day_name", T.StringType(), True),\n                T.StructField("is_weekend", T.BooleanType(), True),\n            ]\n        ),\n    )\n\n    return fallback_row.unionByName(dim_date)\n\n\n# ------------------------------------------------------------\n# Build the product dimension at the product/sub-product grain.\n# This gives Gold a stable surrogate key for complaint product\n# analysis while preserving the business labels from Silver.\n# ------------------------------------------------------------\ndef build_dim_product(silver_dataframe: DataFrame) -> DataFrame:\n    """Build the product dimension at the product and sub-product grain."""\n\n    base_dimension = (\n        silver_dataframe\n        .select("product", "sub_product")\n        .distinct()\n        .withColumn(\n            "_sort_product",\n            F.coalesce(F.col("product"), F.lit(MISSING_MEMBER_LABEL)),\n        )\n        .withColumn(\n            "sub_product",\n            F.coalesce(F.col("sub_product"), F.lit(MISSING_MEMBER_LABEL)),\n        )\n        .withColumn(\n            "_sort_sub_product",\n            F.coalesce(F.col("sub_product"), F.lit(MISSING_MEMBER_LABEL)),\n        )\n        .withColumn(\n            "product_key",\n            F.row_number().over(\n                Window.orderBy("_sort_product", "_sort_sub_product")\n            ),\n        )\n        .drop("_sort_product", "_sort_sub_product")\n        .select("product_key", "product", "sub_product")\n    )\n\n    fallback_row = spark_single_row(\n        silver_dataframe.sparkSession,\n        {\n            "product_key": 0,\n            "product": MISSING_MEMBER_LABEL,\n            "sub_product": MISSING_MEMBER_LABEL,\n        },\n        schema=T.StructType(\n            [\n                T.StructField("product_key", T.IntegerType(), False),\n                T.StructField("product", T.StringType(), True),\n                T.StructField("sub_product", T.StringType(), True),\n            ]\n        ),\n    )\n\n    return fallback_row.unionByName(base_dimension)\n\n\n# ------------------------------------------------------------\n# Build the company dimension so complaint facts can join to a\n# reusable organisation lookup instead of repeating company text\n# in every downstream aggregate model.\n# ------------------------------------------------------------\ndef build_dim_company(silver_dataframe: DataFrame) -> DataFrame:\n    """Build the company dimension."""\n\n    base_dimension = (\n        silver_dataframe\n        .select("company")\n        .distinct()\n        .withColumn(\n            "_sort_company",\n            F.coalesce(F.col("company"), F.lit(MISSING_MEMBER_LABEL)),\n        )\n        .withColumn(\n            "company_key",\n            F.row_number().over(Window.orderBy("_sort_company")),\n        )\n        .drop("_sort_company")\n        .select("company_key", "company")\n    )\n\n    fallback_row = spark_single_row(\n        silver_dataframe.sparkSession,\n        {\n            "company_key": 0,\n            "company": MISSING_MEMBER_LABEL,\n        },\n        schema=T.StructType(\n            [\n                T.StructField("company_key", T.IntegerType(), False),\n                T.StructField("company", T.StringType(), True),\n            ]\n        ),\n    )\n\n    return fallback_row.unionByName(base_dimension)\n\n\n# ------------------------------------------------------------\n# Build the state dimension from the cleaned Silver geography\n# field. A dedicated dimension supports state-level analytics\n# and keeps null or missing states mapped to a single fallback\n# member.\n# ------------------------------------------------------------\ndef build_dim_state(silver_dataframe: DataFrame) -> DataFrame:\n    """Build the state dimension from the cleaned Silver geography fields."""\n\n    base_dimension = (\n        silver_dataframe\n        .select("state")\n        .withColumn(\n            "state",\n            F.coalesce(F.col("state"), F.lit(MISSING_MEMBER_LABEL)),\n        )\n        .distinct()\n        .withColumn(\n            "_sort_state",\n            F.coalesce(F.col("state"), F.lit(MISSING_MEMBER_LABEL)),\n        )\n        .withColumn(\n            "state_key",\n            F.row_number().over(Window.orderBy("_sort_state")),\n        )\n        .drop("_sort_state")\n        .select("state_key", "state")\n    )\n\n    fallback_row = spark_single_row(\n        silver_dataframe.sparkSession,\n        {\n            "state_key": 0,\n            "state": MISSING_MEMBER_LABEL,\n        },\n        schema=T.StructType(\n            [\n                T.StructField("state_key", T.IntegerType(), False),\n                T.StructField("state", T.StringType(), True),\n            ]\n        ),\n    )\n\n    return fallback_row.unionByName(base_dimension)\n\n\n# ------------------------------------------------------------\n# Build the complaint fact table by joining Silver complaints to\n# each conformed dimension and preserving the operational and\n# lineage fields needed for analytics and traceability.\n# ------------------------------------------------------------\ndef build_fact_consumer_complaints(\n    silver_dataframe: DataFrame,\n    dim_date: DataFrame,\n    dim_product: DataFrame,\n    dim_company: DataFrame,\n    dim_state: DataFrame,\n) -> DataFrame:\n    """Build the complaint fact table by joining Silver to the conformed dimensions."""\n\n    received_date_dimension = (\n        dim_date\n        .select(\n            F.col("date_key").alias("received_date_key"),\n            F.col("full_date").alias("received_full_date"),\n        )\n    )\n\n    sent_date_dimension = (\n        dim_date\n        .select(\n            F.col("date_key").alias("sent_date_key"),\n            F.col("full_date").alias("sent_full_date"),\n        )\n    )\n\n    fact_dataframe = (\n        silver_dataframe.alias("s")\n        .join(\n            dim_product.alias("p"),\n            on=[\n                silver_dataframe["product"].eqNullSafe(dim_product["product"]),\n                silver_dataframe["sub_product"].eqNullSafe(dim_product["sub_product"]),\n            ],\n            how="left",\n        )\n        .join(\n            dim_company.alias("c"),\n            on=silver_dataframe["company"].eqNullSafe(dim_company["company"]),\n            how="left",\n        )\n        .join(\n            dim_state.alias("st"),\n            on=silver_dataframe["state"].eqNullSafe(dim_state["state"]),\n            how="left",\n        )\n        .join(\n            received_date_dimension.alias("dr"),\n            on=F.to_date(silver_dataframe["date_received"]) == received_date_dimension["received_full_date"],\n            how="left",\n        )\n        .join(\n            sent_date_dimension.alias("ds"),\n            on=F.to_date(silver_dataframe["date_sent_to_company"]) == sent_date_dimension["sent_full_date"],\n            how="left",\n        )\n        .select(\n            F.col("s.complaint_id"),\n            F.coalesce(F.col("dr.received_date_key"), F.lit(0)).alias("received_date_key"),\n            F.coalesce(F.col("ds.sent_date_key"), F.lit(0)).alias("sent_date_key"),\n            F.coalesce(F.col("p.product_key"), F.lit(0)).alias("product_key"),\n            F.coalesce(F.col("c.company_key"), F.lit(0)).alias("company_key"),\n            F.coalesce(F.col("st.state_key"), F.lit(0)).alias("state_key"),\n            F.col("s.issue"),\n            F.col("s.sub_issue"),\n            F.col("s.zip_code"),\n            F.col("s.tags"),\n            F.col("s.submitted_via"),\n            F.col("s.company_response_to_consumer"),\n            F.col("s.timely_response"),\n            F.when(F.col("s.consumer_complaint_narrative").isNotNull(), F.lit(True)).otherwise(F.lit(False)).alias("has_consumer_narrative"),\n            F.when(F.col("s.tags").isNotNull(), F.lit(True)).otherwise(F.lit(False)).alias("has_tags"),\n            F.when(F.col("s.date_sent_to_company").isNotNull(), F.lit(True)).otherwise(F.lit(False)).alias("has_company_response_date"),\n            F.col("s._ingestion_date"),\n            F.col("s._ingested_at"),\n            F.col("s._source_zip_path"),\n            F.col("s._source_csv_name"),\n            F.col("s._silver_processed_at"),\n            F.col("s._silver_record_status"),\n            F.current_timestamp().alias("_gold_processed_at"),\n        )\n    )\n\n    return fact_dataframe\n\n\n# ------------------------------------------------------------\n# Create a single-row Spark DataFrame with a controlled schema.\n# This is used for explicit fallback members in Gold dimensions\n# so foreign keys always have a safe fallback value.\n# ------------------------------------------------------------\ndef spark_single_row(\n    spark: SparkSession,\n    row: dict[str, object],\n    schema: T.StructType,\n) -> DataFrame:\n    """Create a one-row Spark DataFrame with a specific schema."""\n\n    return spark.createDataFrame([row], schema=schema)\n\n\n# ------------------------------------------------------------\n# Write any Gold table using an idempotent full overwrite. Gold\n# is modelled as a refreshed analytical layer rather than an\n# append-only raw ingestion surface.\n# ------------------------------------------------------------\ndef write_table(dataframe: DataFrame, target_table: str) -> None:\n    """Overwrite a Gold Delta table idempotently."""\n\n    LOGGER.info("Writing Gold table to %s", target_table)\n\n    (\n        dataframe.write\n        .format("delta")\n        .mode("overwrite")\n        .option("overwriteSchema", "true")\n        .saveAsTable(target_table)\n    )\n\n\n# ------------------------------------------------------------\n# Validate that the dimensional model is complete after the\n# write: dimensions must be populated, the fact grain must match\n# Silver complaint grain, and no dimension keys may be null.\n# ------------------------------------------------------------\ndef validate_gold_tables(\n    spark: SparkSession,\n    silver_row_count: int,\n) -> None:\n    """Run post-write validations across the Gold dimensional model."""\n\n    dim_date_count = spark.table(DIM_DATE_TABLE).count()\n    dim_product_count = spark.table(DIM_PRODUCT_TABLE).count()\n    dim_company_count = spark.table(DIM_COMPANY_TABLE).count()\n    dim_state_count = spark.table(DIM_STATE_TABLE).count()\n    fact_row_count = spark.table(FACT_TABLE).count()\n\n    if min(dim_date_count, dim_product_count, dim_company_count, dim_state_count) <= 0:\n        raise RuntimeError("Gold validation failed: one or more dimensions are empty.")\n\n    if fact_row_count == 0:\n        raise RuntimeError("Gold validation failed: fact table row count is zero.")\n\n    if fact_row_count != silver_row_count:\n        raise RuntimeError(\n            "Gold validation failed: fact row count does not match the Silver complaint grain."\n        )\n\n    duplicate_complaint_ids = (\n        spark.table(FACT_TABLE)\n        .groupBy("complaint_id")\n        .count()\n        .filter(F.col("count") > 1)\n        .count()\n    )\n    if duplicate_complaint_ids > 0:\n        raise RuntimeError(\n            f"Gold validation failed: {duplicate_complaint_ids} duplicate complaint IDs found in the fact table."\n        )\n\n    null_dimension_keys = (\n        spark.table(FACT_TABLE)\n        .filter(\n            F.col("product_key").isNull()\n            | F.col("company_key").isNull()\n            | F.col("state_key").isNull()\n            | F.col("received_date_key").isNull()\n            | F.col("sent_date_key").isNull()\n        )\n        .count()\n    )\n    if null_dimension_keys > 0:\n        raise RuntimeError(\n            f"Gold validation failed: {null_dimension_keys} fact rows contain null dimension keys."\n        )\n\n    LOGGER.info(\n        "Gold validation successful. Dimensions: date=%s, product=%s, company=%s, state=%s. Fact rows=%s.",\n        f"{dim_date_count:,}",\n        f"{dim_product_count:,}",\n        f"{dim_company_count:,}",\n        f"{dim_state_count:,}",\n        f"{fact_row_count:,}",\n    )\n\n\n# ------------------------------------------------------------\n# Run the end-to-end Gold warehouse build in dependency order:\n# validate Silver, build dimensions, build the complaint fact,\n# write all Gold tables, then run final validation checks.\n# ------------------------------------------------------------\ndef main() -> None:\n    """Run the full Gold warehouse build for consumer complaints."""\n\n    spark = SparkSession.builder.getOrCreate()\n\n    LOGGER.info("Starting Consumer Complaints Gold processing.")\n\n    silver_dataframe, silver_row_count = validate_source_table(spark)\n    ensure_gold_schema_exists(spark)\n\n    dim_date = build_dim_date(silver_dataframe)\n    dim_product = build_dim_product(silver_dataframe)\n    dim_company = build_dim_company(silver_dataframe)\n    dim_state = build_dim_state(silver_dataframe)\n    fact_consumer_complaints = build_fact_consumer_complaints(\n        silver_dataframe=silver_dataframe,\n        dim_date=dim_date,\n        dim_product=dim_product,\n        dim_company=dim_company,\n        dim_state=dim_state,\n    )\n\n    write_table(dim_date, DIM_DATE_TABLE)\n    write_table(dim_product, DIM_PRODUCT_TABLE)\n    write_table(dim_company, DIM_COMPANY_TABLE)\n    write_table(dim_state, DIM_STATE_TABLE)\n    write_table(fact_consumer_complaints, FACT_TABLE)\n\n    validate_gold_tables(spark, silver_row_count)\n\n    LOGGER.info("Consumer Complaints Gold processing finished successfully.")\n\n\nif __name__ == "__main__":\n    main()\n